# V1DD stimulus metrics — pre-flight

A read-only survey of **every** session in the functional asset, run before the pipeline
is widened from the two coregistered sessions to all of them.

It exists because three things the widened pipeline depends on are currently assumptions
rather than facts:

**Does every session carry all six stimuli?** `stimulus_trials` returns an *empty frame*
for a missing stimulus rather than raising, so a session that never ran a given stimulus
would flow silently into the metric functions and produce confident nonsense. Both
validated sessions are 2p with all seven blocks; the 3p sessions have never been looked at.

**Where does the mouse id live?** The tables hard-code `M409828`. `nwb.subject.subject_id`
is the obvious source but nothing has read it, so this records what is actually there
beside the id parsed from the session directory name, and whether they agree.

**Do the imaging planes carry depth?** The ROI tables carry `column, volume, plane, roi`
and no physical depth, which makes tuning-versus-depth unaskable. If the `ImagingPlane`
holds it, it belongs in the output schema.

Nothing here loads a trace array — `load_plane` pulls every sample for every ROI, roughly
400 MB per session, which is fine for two and not for thirty-five. Shapes and timestamps
are metadata and cost nothing.

Two artifacts land in the validation directory, **not** in the asset: a one-row-per-session
`preflight_summary.csv` and a `preflight_report.json` holding the aggregate verdict plus
full detail for one session per column.

In [ ]:
import os
import sys
from os.path import join as pjoin

import numpy as np
import pandas as pd
from IPython.display import display

for _c in [pjoin("..", "utils"), pjoin("code", "utils"), "utils"]:
    if os.path.isdir(_c):
        sys.path.append(os.path.abspath(_c))
        break
for _c in [".", pjoin("..", "validation"), pjoin("code", "validation")]:
    if os.path.isfile(pjoin(_c, "preflight.py")):
        sys.path.append(os.path.abspath(_c))
        break
else:
    raise FileNotFoundError(f"could not locate preflight.py; cwd={os.getcwd()}")

import preflight as pf
from paths import resolve_data_root, resolve_dataset_dir

pd.set_option("display.max_columns", None)
pd.set_option("display.max_rows", 100)
print(f"numpy {np.__version__} | pandas {pd.__version__}")

In [ ]:
# The mounted asset is the only configured input. No materialization version: nothing in
# this pipeline touches CAVE, and the mouse comes out of the data rather than a constant.
functional_asset = "409828_V1DD_Filtered"

data_root = resolve_data_root(functional_asset)
functional_dir = resolve_dataset_dir(functional_asset, root=data_root)

output_target = "scratch"          # validation artifacts are never part of the asset
save_dir = pjoin(f"/{output_target}", "v1dd_stimulus_metrics_validation")
os.makedirs(save_dir, exist_ok=True)

# Set to a small number for a smoke test before committing to the full sweep.
LIMIT = None

print(f"functional_dir : {functional_dir}")
print(f"save_dir       : {save_dir}")
print(f"sessions found : {len(pf.vn.find_sessions(functional_dir))}")

In [ ]:
%%time
# One dot per stimulus present, one x per stimulus absent, in the order
# DGF DGW NI NI12 NM LSN -- so a session with a gap is visible as it goes past.
verdict = pf.run_preflight(functional_dir, save_dir, limit=LIMIT)

In [ ]:
summary = pd.read_csv(pjoin(save_dir, "preflight_summary.csv"), dtype={"volume": str})
display(summary)

In [ ]:
# --- Question 1: does every session carry all six stimuli?
print("STIMULUS COVERAGE")
print(f"  {verdict['sessions_with_all_six']} of {verdict['n_readable']} readable sessions "
      f"have all six families")
for fam, missing in verdict["sessions_missing_a_family"].items():
    if missing:
        print(f"  {fam}: absent in {len(missing)} session(s) -> {missing[:6]}")
if verdict["sessions_without_spontaneous"]:
    # No spontaneous block means no bootstrap null, which means no z_score,
    # no is_responsive and no receptive fields for that session.
    print(f"  !! no spontaneous block: {verdict['sessions_without_spontaneous']}")
print(f"  LSN template usable in {verdict['lsn_template_ok']} session(s) "
      f"(gates receptive fields)")
if verdict["unreadable"]:
    print(f"  !! unreadable: {verdict['unreadable']}")

# --- Question 2: where does the mouse id live?
print()
print("MOUSE")
print(f"  from nwb.subject : {verdict['mouse_from_subject'] or 'ABSENT'}")
print(f"  from directory   : {verdict['mouse_from_name'] or 'ABSENT'}")
print(f"  sources agree    : {verdict['mouse_sources_agree']}")

# --- Question 3: do the imaging planes carry depth?
print()
print("IMAGING PLANE (depth candidates)")
print(f"  locations: {verdict['imaging_plane_locations']}")
for col, rec in verdict.get("detail_samples", {}).items():
    detail = ((rec.get("planes") or {}).get("detail") or {})
    print(f"  column {col}:")
    for k, d in list(detail.items())[:3]:
        ip = d.get("imaging_plane") or {}
        print(f"    {k}: location={ip.get('location')!r} "
              f"origin_coords={ip.get('origin_coords')} "
              f"({ip.get('origin_coords_unit')}) desc={str(ip.get('description'))[:40]!r}")

print()
print("SHAPE OF THE ASSET")
for k in ("formats", "columns", "volumes", "n_planes", "n_direction_values"):
    print(f"  {k:<20} {verdict[k]}")
print(f"  {'dt_range':<20} {verdict['dt_range']}")

## What happens next

The summary CSV and the JSON verdict are committed from `/scratch` and read away from the
capsule. Three outcomes drive the refactor:

* **Any session missing a family** means each metric family needs a skip-with-NaN path and
  a coverage column in the output, rather than assuming the stimulus is there.
* **Whichever rung of the mouse ladder carries the id** becomes the derivation, replacing
  the hard-coded `M409828`. If the two sources disagree anywhere, that session is named
  above and needs a look before either is trusted.
* **If `origin_coords` or `location` carries depth**, `depth_um` joins the ROI identity
  block — which changes `OUTPUT_COLUMNS` and the column-order test, so it lands with the
  other schema work rather than in the mechanical refactor.